# End-to-End RAG Pipeline Smoke Test

## Goal

Run one user prompt through the project components that exist today:

1. Groq plans discovery keywords.
2. alphaXiv MCP discovers papers, with an optional ArXiv fallback.
3. ArXiv PDFs are downloaded into category directories.
4. The ingestion pipeline parses, cleans, sections, and persists each paper.
5. The processing pipeline chunks, embeds, builds BM25, and indexes Qdrant.
6. Dense and sparse retrieval find evidence for the original prompt.
7. A minimal rank-fusion step combines the evidence and Groq produces a grounded answer.

This is a bounded live smoke test, not a bulk ingestion job. It uses isolated artifacts under `data/e2e_smoke_test/` and a dedicated Qdrant test collection so production indexes are not replaced.

## Setup

In [1]:
import json
import os
import shlex
import sys
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve() if (Path.cwd() / "ingestion").is_dir() else Path.cwd().resolve().parent
if not (REPO_ROOT / "ingestion").is_dir():
    raise RuntimeError("Run this notebook from the repository root")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

from ingestion.paper_discovery import build_discovery
from ingestion.pipeline import IngestionPipeline
from processing.bm25_indexer import BM25Indexer
from processing.chunker import SectionAwareChunker
from processing.embedder import Embedder
from processing.pipeline import ProcessingPipeline
from processing.qdrant_indexer import QdrantIndexer
from qdrant_client import QdrantClient
from retrieval.dense_retriever import DenseRetriever
from retrieval.query_processor import QueryProcessor
from retrieval.sparse_retriever import SparseRetriever

D:\data science\project\RAG-AI_Reasearch_Papers\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


### Parameters

In [2]:
USER_PROMPT = "How does retrieval-augmented generation improve factual accuracy?"
MAX_PAPERS = 2
TOP_K = 5
REQUIRE_MCP = True
RECREATE_QDRANT = True

DATA_DIR = REPO_ROOT / "data" / "e2e_smoke_test"
BM25_INDEX_PATH = DATA_DIR / "processed" / "bm25_index.pkl"
QDRANT_URL = f"http://{os.getenv('QDRANT_HOST', 'localhost')}:{os.getenv('QDRANT_PORT', '6333')}"
QDRANT_COLLECTION = f"{os.getenv('QDRANT_COLLECTION', 'ai_papers')}_e2e_test"
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

assert USER_PROMPT.strip()
assert MAX_PAPERS > 0 and TOP_K > 0
assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in .env"

pd.Series({
    "prompt": USER_PROMPT,
    "max_papers": MAX_PAPERS,
    "require_mcp": REQUIRE_MCP,
    "qdrant_url": QDRANT_URL,
    "qdrant_collection": QDRANT_COLLECTION,
    "embedding_model": EMBEDDING_MODEL,
    "groq_model": GROQ_MODEL,
})

prompt               How does retrieval-augmented generation improv...
max_papers                                                           2
require_mcp                                                       True
qdrant_url                                       http://localhost:6333
qdrant_collection                                   ai_papers_e2e_test
embedding_model                                       all-MiniLM-L6-v2
groq_model                                     llama-3.3-70b-versatile
dtype: object

## Steps

### 1. Confirm Qdrant is reachable

In [3]:
qdrant_client = QdrantClient(url=QDRANT_URL, timeout=10)
existing_collections = [item.name for item in qdrant_client.get_collections().collections]
print("Qdrant reachable. Existing collections:", existing_collections)

Qdrant reachable. Existing collections: []


### 2. Run discovery and ingestion

In [4]:
discovery = build_discovery("auto")
ingestion = IngestionPipeline(
    data_dir=DATA_DIR,
    discovery=discovery,
    discovery_provider="auto",
)
ingestion_result = ingestion.run(USER_PROMPT, max_results=MAX_PAPERS)

provider_used = getattr(discovery, "last_provider_used", "unknown")
primary = getattr(discovery, "primary", None)
planner_attempts = getattr(primary, "last_arguments", None) or []

print("Discovery provider used:", provider_used)
display(pd.DataFrame(planner_attempts))
display(pd.Series(ingestion_result.to_dict()).drop("errors"))
if ingestion_result.errors:
    display(pd.DataFrame(ingestion_result.errors))

assert ingestion_result.processed > 0, "No paper completed ingestion"
if REQUIRE_MCP:
    assert provider_used == "alphaxiv-mcp", (
        "MCP discovery did not complete; fallback was used. "
        f"Reason: {getattr(discovery, 'last_primary_error', None)}"
    )
assert any(plan.get("planner") == "groq" for plan in planner_attempts), (
    "Groq did not produce a usable discovery plan"
)

Discovery provider used: alphaxiv-mcp


,keywords,question,difficulty,planner
0,"[retrieval-augmented generation, factual accur...",How does retrieval-augmented generation improv...,4,groq
1,"[information retrieval, language generation]",What role does information retrieval play in i...,5,groq
2,"[retrieval-augmented generation, does, retriev...",Find ranked research papers that best match th...,5,NaN
3,[How does retrieval-augmented generation impro...,"Find concrete research papers, not a prose sum...",7,NaN


query                 How does retrieval-augmented generation improv...
discovery_provider                                                 auto
discovered                                                            2
downloaded                                                            1
skipped                                                               1
failed                                                                0
processed                                                             2
metadata_path         D:\data science\project\RAG-AI_Reasearch_Paper...
processed_dir         D:\data science\project\RAG-AI_Reasearch_Paper...
dtype: object

### 3. Verify category routing and processed documents

In [5]:
metadata_records = json.loads(Path(ingestion_result.metadata_path).read_text(encoding="utf-8"))
processed_root = Path(ingestion_result.processed_dir)
ingested_rows = []
processed_paths = []

for record in metadata_records:
    category = str(record.get("primary_category") or "unknown").replace("/", "_").replace("\\", "_")
    paper_id = str(record["paper_id"]).replace("/", "_").replace("\\", "_")
    pdf_path = Path(record["local_pdf_path"])
    processed_path = processed_root / category / f"{paper_id}.json"
    processed_paths.append(processed_path)
    ingested_rows.append({
        "paper_id": record["paper_id"],
        "category": category,
        "pdf_exists": pdf_path.is_file(),
        "pdf_path": str(pdf_path),
        "processed_exists": processed_path.is_file(),
        "processed_path": str(processed_path),
    })

routing_frame = pd.DataFrame(ingested_rows)
display(routing_frame)
assert len(routing_frame) == ingestion_result.processed
assert routing_frame[["pdf_exists", "processed_exists"]].all().all()
assert all(Path(row.pdf_path).parent.name == row.category for row in routing_frame.itertuples())

,paper_id,category,pdf_exists,pdf_path,processed_exists,processed_path
0,2604.22843,unknown,True,D:\data science\project\RAG-AI_Reasearch_Paper...,True,D:\data science\project\RAG-AI_Reasearch_Paper...
1,2605.17989,unknown,True,D:\data science\project\RAG-AI_Reasearch_Paper...,True,D:\data science\project\RAG-AI_Reasearch_Paper...


### 4. Chunk, embed, and index

In [6]:
embedder = Embedder(model_name=EMBEDDING_MODEL)
bm25_indexer = BM25Indexer()
qdrant_indexer = QdrantIndexer(
    client=qdrant_client,
    collection_name=QDRANT_COLLECTION,
)
processing = ProcessingPipeline(
    chunker=SectionAwareChunker(max_tokens=512, overlap_tokens=80),
    embedder=embedder,
    bm25_indexer=bm25_indexer,
    qdrant_indexer=qdrant_indexer,
)
processing_result = processing.process_paths(
    processed_paths,
    index_dense=True,
    recreate_qdrant=RECREATE_QDRANT,
)
bm25_indexer.save(BM25_INDEX_PATH)

processing_summary = {
    "papers": len(processed_paths),
    "chunks": len(processing_result["chunks"]),
    "embeddings": len(processing_result["embedding_records"]),
    "bm25_documents": processing_result["bm25_documents"],
    "qdrant_points_upserted": processing_result["qdrant_points"],
    "bm25_index_path": str(BM25_INDEX_PATH),
}
display(pd.Series(processing_summary))

assert processing_summary["chunks"] > 0
assert processing_summary["chunks"] == processing_summary["embeddings"]
assert processing_summary["chunks"] == processing_summary["bm25_documents"]
assert processing_summary["chunks"] == processing_summary["qdrant_points_upserted"]

papers                                                                    2
chunks                                                                  267
embeddings                                                              267
bm25_documents                                                          267
qdrant_points_upserted                                                  267
bm25_index_path           D:\data science\project\RAG-AI_Reasearch_Paper...
dtype: object

### 5. Retrieve with dense and BM25 search

In [7]:
sparse_retriever = SparseRetriever(BM25_INDEX_PATH, default_top_k=TOP_K)
query_processor = QueryProcessor(
    enable_expansion=True,
    preprocessing_config=sparse_retriever.preprocessing_config,
)
processed_query = query_processor.process(USER_PROMPT)

dense_retriever = DenseRetriever(
    qdrant_client,
    embedder,
    QDRANT_COLLECTION,
    default_top_k=TOP_K,
)
dense_results = dense_retriever.search(processed_query.dense_query)
sparse_results = sparse_retriever.search(processed_query.sparse_tokens)

def result_rows(results):
    return [{
        "source": item.source,
        "score": item.score,
        "paper_id": item.paper_id,
        "section": item.section,
        "title": item.title,
        "text": item.text[:220],
    } for item in results]

print("Processed query:", processed_query)
display(pd.DataFrame(result_rows(dense_results)))
display(pd.DataFrame(result_rows(sparse_results)))
assert dense_results and sparse_results

Processed query: ProcessedQuery(original_query='How does retrieval-augmented generation improve factual accuracy?', cleaned_query='How does retrieval-augmented generation improve factual accuracy?', dense_query='How does retrieval-augmented generation improve factual accuracy?', sparse_tokens=['how', 'does', 'retrieval-augmented', 'generation', 'improve', 'factual', 'accuracy'], expanded_query='How does retrieval-augmented generation improve factual accuracy?')


,source,score,paper_id,section,title,text
0,dense,0.683734,2604.22843,front_matter,alphaXiv paper 2604.22843,Structure Guided Retrieval-Augmented Generatio...
1,dense,0.667076,2605.17989,introduction,alphaXiv paper 2605.17989,Retrieval-Augmented Generation (RAG) has becom...
2,dense,0.660367,2605.17989,abstract,alphaXiv paper 2605.17989,Retrieval-Augmented Generation (RAG) im- prove...
3,dense,0.597842,2605.17989,methodology,alphaXiv paper 2605.17989,different problem than ours. Even with optimiz...
4,dense,0.590596,2605.17989,conclusion,alphaXiv paper 2605.17989,We presented a predictive asynchronous retriev...


,source,score,paper_id,section,title,text
0,sparse,11.361493,2605.17989,methodology,alphaXiv paper 2605.17989,multiple valid paths. Our query construction a...
1,sparse,7.800984,2604.22843,abstract,alphaXiv paper 2604.22843,Retrieval-Augmented Generation (RAG) has been ...
2,sparse,7.538529,2605.17989,methodology,alphaXiv paper 2605.17989,approach achieves 35.6 ROUGE-1 while reducing ...
3,sparse,7.273912,2605.17989,methodology,alphaXiv paper 2605.17989,extended wait (4 to 5 tokens): complex logical...
4,sparse,6.955576,2605.17989,experiments,alphaXiv paper 2605.17989,We design experiments to answer three question...


### 6. Fuse the rankings and generate a grounded answer

In [8]:
fused = {}
for results in (dense_results, sparse_results):
    for rank, item in enumerate(results, start=1):
        entry = fused.setdefault(item.chunk_id, {"result": item, "rrf_score": 0.0})
        entry["rrf_score"] += 1.0 / (60 + rank)

fused_results = sorted(fused.values(), key=lambda item: item["rrf_score"], reverse=True)[:TOP_K]
context_blocks = []
for rank, entry in enumerate(fused_results, start=1):
    item = entry["result"]
    citation = f"[{item.paper_id}, {item.section or 'unknown section'}]"
    context_blocks.append(f"Source {rank} {citation}\n{item.text}")

context = "\n\n".join(context_blocks)
response = httpx.post(
    "https://api.groq.com/openai/v1/chat/completions",
    headers={"authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
    json={
        "model": GROQ_MODEL,
        "temperature": 0.1,
        "messages": [
            {
                "role": "system",
                "content": (
                    "Answer only from the supplied research-paper context. "
                    "Cite claims as [paper_id, section]. If the evidence is insufficient, say so."
                ),
            },
            {"role": "user", "content": f"Question: {USER_PROMPT}\n\nContext:\n{context}"},
        ],
    },
    timeout=60,
)
response.raise_for_status()
answer = response.json()["choices"][0]["message"]["content"].strip()
print(answer)
assert answer

Retrieval-augmented generation improves factual accuracy by mitigating hallucination and knowledge staleness in large language models [2605.17989, introduction]. It achieves this by augmenting static model parameters with external knowledge, which effectively reduces the likelihood of generating factually incorrect information [2604.22843, abstract]. 

Structure-guided retrieval-augmented generation (SG-RAG) is a specific approach that models the retrieval process as an embedding-based subgraph matching task, using retrieved topological structures to guide the language model to generate answers that meet all specified query conditions [2604.22843, abstract]. This approach has been shown to significantly outperform strong baselines, delivering absolute gains of 20.68–50.88 percentage points [2604.22843, abstract].

Additionally, predictive prefetching for retrieval-augmented generation can also improve factual accuracy by predicting when retrieval should be triggered and what informatio

## Checks

In [9]:
health = dense_retriever.health_check()
checks = {
    "groq_discovery_plan": any(plan.get("planner") == "groq" for plan in planner_attempts),
    "mcp_provider_used": provider_used == "alphaxiv-mcp",
    "papers_ingested": ingestion_result.processed > 0,
    "category_routing_valid": routing_frame[["pdf_exists", "processed_exists"]].all().all(),
    "chunks_created": processing_summary["chunks"] > 0,
    "bm25_saved": BM25_INDEX_PATH.is_file(),
    "qdrant_collection_exists": health["collection_exists"],
    "qdrant_has_points": bool(health.get("points_count")),
    "embedding_dimension_matches": health.get("dimension_match") is True,
    "dense_results_returned": bool(dense_results),
    "sparse_results_returned": bool(sparse_results),
    "answer_generated": bool(answer),
}
display(pd.Series(checks, name="passed"))
assert all(checks.values()), "One or more end-to-end checks failed"

groq_discovery_plan            True
mcp_provider_used              True
papers_ingested                True
category_routing_valid         True
chunks_created                 True
bm25_saved                     True
qdrant_collection_exists       True
qdrant_has_points              True
embedding_dimension_matches    True
dense_results_returned         True
sparse_results_returned        True
answer_generated               True
Name: passed, dtype: bool

## Next Steps

If all checks are `True`, the implemented pipeline completed end to end for the selected prompt. The fusion and final Groq call in this notebook are deliberately small adapters; they should move into production modules when hybrid retrieval and answer generation are implemented in the application.